# Laboratório 10 — Pipeline Definitivo

**QLoRA (4-bit) + RAG massivo + KV Cache + FlashAttention-2**

Execução prevista no Google Colab Free (GPU T4, 15GB VRAM).

## Setup

Instalação das dependências e imports base.

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets sentencepiece matplotlib

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
assert torch.cuda.is_available(), "GPU CUDA é obrigatória (use Colab com runtime GPU)"
print("GPU:", torch.cuda.get_device_name(0))

## Passo 1 — Ingestão eficiente (QLoRA 4-bit)

Carregamento do modelo base em 4 bits via `bitsandbytes` para reduzir o footprint inicial de VRAM.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

In [ ]:
torch.cuda.synchronize()
vram_modelo_mb = torch.cuda.memory_allocated() / 1024**2
print(f"VRAM ocupada pelo modelo quantizado: {vram_modelo_mb:.1f} MB")